# JOILang PromptOps: Local Model Benchmark & Render Trace

이 노트북은 JOILang-Server 저장소의 `run_benchmark.py`를 활용하여 실제 로컬 모델 환경에서 blocks 기반 프롬프트 렌더링이 어떻게 동작하는지 한 단계씩 점검하기 위한 디버깅 및 트레이싱 도구입니다.

주요 목적:
1. 로컬 환경 변수 설정 및 단일 행(Row 1) 테스트 파이프라인 정상 구동 확인
2. blocks 렌더링과 legacy_v13_monolith 렌더링 비교
3. Retrieval Pre-mapping 옵션에 따른 동작 확인
4. 생성된 결과 아티팩트(`row_comparison.csv` 등)를 로드하여 검증 결과(DET 점수, 지연 시간, 토큰 수 등) 확인

> 참고: 이 노트북은 전체 모델 비교 실험을 돌리기 위한 것이 아니라, 본격적인 대규모 실험 전 환경과 프롬프트 렌더링 파이프라인을 점검하는 용도입니다.

## Section 1. 환경 변수 및 경로 설정
제공된 가이드에 따라 Conda 가상환경 파이썬과 타겟 GPU 디바이스를 설정합니다.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display

# 작업 디렉토리 기준 경로 설정
NOTEBOOK_DIR = Path.cwd().resolve()
VERSION_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = VERSION_ROOT.parent.parent

# 환경 변수 설정 (로컬 환경에 맞게 수정 가능)
os.environ["JOI_V15_WORKER_PYTHON"] = "/home/mgjeong/miniconda3/envs/l/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"

# 실행할 파이썬 스크립트 경로
RUN_BENCHMARK_SCRIPT = VERSION_ROOT / "scripts" / "run_benchmark.py"

# 벤치마크 CLI를 실행할 Python. 보통 현재 Jupyter kernel의 Python을 사용합니다.
# 필요하면 아래 값을 직접 수정하세요. 예: "/home/mgjeong/miniconda3/envs/l/bin/python"
BENCHMARK_PYTHON = sys.executable

print("[Paths]")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"VERSION_ROOT: {VERSION_ROOT}")
print(f"RUN_BENCHMARK_SCRIPT: {RUN_BENCHMARK_SCRIPT}")

print("\n[Environment Variables]")
print(f"WORKER_PYTHON: {os.environ.get('JOI_V15_WORKER_PYTHON')}")
print(f"LOCAL_DEVICE: {os.environ.get('JOI_V15_LOCAL_DEVICE')}")
print(f"BENCHMARK_PYTHON: {BENCHMARK_PYTHON}")

if not RUN_BENCHMARK_SCRIPT.exists():
    print(f"\n[Warning] run_benchmark.py를 찾지 못했습니다: {RUN_BENCHMARK_SCRIPT}")

[Paths]
REPO_ROOT: /root/llm/JOILang-Server
VERSION_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413
RUN_BENCHMARK_SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py

[Environment Variables]
WORKER_PYTHON: /home/mgjeong/miniconda3/envs/l/bin/python
LOCAL_DEVICE: cuda:0
BENCHMARK_PYTHON: /usr/bin/python3


## Section 2. 유틸리티 함수: 벤치마크 실행 및 최근 실행 결과 불러오기
벤치마크 스크립트 실행 후 `results/model_suite_<timestamp>` 폴더에 생성되는 CSV를 자동으로 읽어오기 위한 헬퍼 함수입니다.

In [2]:
def run_benchmark(extra_args, *, cwd=None):
    """run_benchmark.py를 subprocess로 실행합니다."""
    if not RUN_BENCHMARK_SCRIPT.exists():
        raise FileNotFoundError(f"run_benchmark.py not found: {RUN_BENCHMARK_SCRIPT}")

    cmd = [BENCHMARK_PYTHON, str(RUN_BENCHMARK_SCRIPT), *map(str, extra_args)]
    print("Running command:")
    print(" ".join(cmd))
    print()

    completed = subprocess.run(
        cmd,
        cwd=str(cwd or VERSION_ROOT),
        env=os.environ.copy(),
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"Benchmark failed with exit code {completed.returncode}")
    return completed


def get_latest_result_dir(version_root):
    results_dir = Path(version_root) / "results"
    if not results_dir.exists():
        return None

    subdirs = [
        d for d in results_dir.iterdir()
        if d.is_dir() and d.name.startswith("model_suite_")
    ]
    if not subdirs:
        return None

    return max(subdirs, key=lambda p: p.stat().st_mtime)


def inspect_latest_row_comparison(version_root):
    latest_dir = get_latest_result_dir(version_root)
    if latest_dir is None:
        print("No results directory found.")
        return None

    csv_path = latest_dir / "row_comparison.csv"
    if not csv_path.exists():
        print(f"row_comparison.csv not found in {latest_dir}")
        return None

    print(f"Loading results from: {csv_path}")
    df = pd.read_csv(csv_path)

    # 핵심 확인 지표 필터링
    preferred_cols = [
        "row_no",
        "model_key",
        "det_score",
        "is_pass",
        "failure_reasons",
        "prompt_tokens",
        "latency",
        "generated_code",
    ]
    target_cols = [col for col in preferred_cols if col in df.columns]
    if not target_cols:
        return df
    return df[target_cols]


## Section 3. [Recommended] 초기 Blocks Render 및 로컬 모델 테스트
가장 목적에 부합하는 권장 진입점입니다. `qwen25_coder_7b`로 Row 1에 대해 blocks 렌더링 및 Retrieval Pre-mapping 옵션을 켜서 실행합니다.

체크리스트:
1. Generated JOICode가 비어 있지 않은가?
2. JSON valid인가?
3. DET 점수는 얼마인가?
4. failure reason이 무엇인가?
5. prompt tokens가 얼마인가?
6. LLM latency가 찍히는가?

In [11]:
#MODEL="phi35_mini"

In [10]:
#MODEL="gemma2_9b_it"  #Gemma2 9B-it의 컨텍스트 한계 8192를 훨씬 넘는 20800 tokens 프롬프트

/home/mgjeong/miniconda3/envs/l/bin/python - <<'PY'
> from transformers import AutoConfig
> 
> model_path = "/home/mgjeong/.cache/huggingface/hub/models--google--gemma-2-9b-it/snapshots/11c9b309abf73637e4b6f9a3fa1e92e615547819"
> cfg = AutoConfig.from_pretrained(model_path, local_files_only=True)
> 
> for key in [
>     "max_position_embeddings",
>     "sliding_window",
>     "rope_theta",
>     "model_type",
>     "architectures",
> ]:
>     print(key, "=", getattr(cfg, key, None))
> PY

huggingface-cli login 

pip install   transformers==4.35.*   sentence-transformers==2.2.2   huggingface_hub==0.19.*



In [ ]:
#MODEL="qwen25_coder_7b"
#MODEL="llama31_8b" 
#MODEL="qwen25_coder_14b"

# Debuding 
# rm -f /tmp/joi_local_llm_worker_debug.log
# ~~
# tail -300 /tmp/joi_local_llm_worker_debug.log

In [15]:
print("Running basic block render test on qwen25_coder_14b (Row 1)...")

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_14b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

Running basic block render test on qwen25_coder_14b (Row 1)...
Running command:
/usr/bin/python3 /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_14b --row-no 1 --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-device cpu --print-mode compare --strict-availability

Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/model_suite_20260514_213815
Rows: 1 | Models: qwen25_coder_14b
 - qwen25_coder_14b (Qwen2.5-Coder-14B-Instruct): avg_det=0.0000, gt_exact=0/1, pass=0/1, avg_latency=0.0000s, warm_p50=0.0000s, cold_load=0.0000s, avg_prompt_tokens=0.0, peak_vram=0.0000GB, gen_errors=1
   top_generation_error: failed to start persistent worker: FileNotFoundError: [Errno 2] No such file or directory: '/home/mgjeong/miniconda3/envs/l/bin/python'
Suite summary CSV: /root/llm/JOILang-Server

CompletedProcess(args=['/usr/bin/python3', '/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py', '--suite', 'paper_local5', '--model-key', 'qwen25_coder_14b', '--row-no', '1', '--candidate-k', '1', '--repair-attempts', '0', '--det-profile', 'strict', '--prompt-render-mode', 'blocks', '--enable-retrieval-premapping', '--retrieval-topk', '10', '--retrieval-device', 'cpu', '--print-mode', 'compare', '--strict-availability'], returncode=0)

In [14]:
import os
from pathlib import Path

LOCAL_MODELS_DIR = Path("/root/llm/JOILang-Server/local_models").resolve()
os.environ["JOI_V15_LOCAL_MODEL_NAME"] = str(LOCAL_MODELS_DIR / "qwen25_coder_7b")
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_7b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

Running command:
/usr/bin/python3 /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_7b --row-no 1 --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-device cpu --print-mode compare --strict-availability



Unavailable models for current runtime: qwen25_coder_7b


RuntimeError: Benchmark failed with exit code 1

In [ ]:
# 위 셀 실행 후 생성된 결과를 데이터프레임으로 확인
df_result_1 = inspect_latest_row_comparison(VERSION_ROOT)
if df_result_1 is not None:
    display(df_result_1)

## Section 4. Full Schema Fallback (Retrieval 끄기) 테스트
Retrieval pre-mapping 없이 전체 스키마를 포함시켜 렌더링합니다. 프롬프트가 길어지므로 토큰 수와 latency 증가를 관찰할 수 있습니다.

In [ ]:
print("Running Full Schema Fallback test...")

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_7b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--disable-retrieval-premapping",
    "--print-mode", "compare",
    "--strict-availability",
])

In [ ]:
df_result_full_schema = inspect_latest_row_comparison(VERSION_ROOT)
if df_result_full_schema is not None:
    display(df_result_full_schema)

## Section 5. Legacy Monolith (v13) vs Blocks (v15) 렌더링 모드 비교
`legacy_v13_monolith` 모드를 실행합니다. 이후 Section 3의 blocks 결과와 비교해볼 수 있습니다.

참고: 이 테스트를 위해서는 `version0_13` 디렉토리에 프롬프트 에셋이 존재해야 합니다.

In [ ]:
print("Running Legacy Monolith test...")

prompt_assets_dir = REPO_ROOT / "gpt_mg" / "version0_13"
print(f"prompt_assets_dir: {prompt_assets_dir}")

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_7b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "legacy_v13_monolith",
    "--prompt-assets-dir", str(prompt_assets_dir),
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

In [ ]:
df_result_legacy = inspect_latest_row_comparison(VERSION_ROOT)
if df_result_legacy is not None:
    display(df_result_legacy)

## Section 6. 스케일업: 카테고리 및 여러 행 실행 예시
Row 1이 성공적으로 실행된다면, 본격적인 진화를 위해 여러 행을 평가할 수 있습니다.
아래 명령은 `--category 1`과 `--category 2`에 대해 카테고리당 3개씩, 총 6개의 데이터를 평가합니다.

실행 시간이 길어질 수 있으므로 기본적으로 주석 처리되어 있습니다. 필요 시 주석을 해제한 뒤 실행하세요.

In [ ]:
# 필요 시 아래 블록의 주석을 해제하고 실행하세요.
#
# print("Running multi-row category test...")
# run_benchmark([
#     "--suite", "paper_local5",
#     "--model-key", "qwen25_coder_7b",
#     "--category", "1",
#     "--category", "2",
#     "--limit-per-category", "3",
#     "--candidate-k", "1",
#     "--repair-attempts", "0",
#     "--det-profile", "strict",
#     "--prompt-render-mode", "blocks",
#     "--enable-retrieval-premapping",
#     "--retrieval-topk", "10",
#     "--retrieval-device", "cpu",
#     "--print-mode", "summary",
#     "--strict-availability",
# ])

print("주석을 해제하면 multi-row test를 실행할 수 있습니다.")

## Section 7. 결과 아티팩트 디렉토리 점검
가장 최근 실행된 결과 디렉토리 내에 HTML/PDF 리포트 및 각종 요약 CSV가 정상적으로 생성되었는지 확인합니다.

In [ ]:
latest_dir = get_latest_result_dir(VERSION_ROOT)
if latest_dir:
    print(f"Latest Result Directory: {latest_dir.name}\n")
    print("--- Contained Files & Reports ---")
    for item in latest_dir.rglob("*"):
        if item.is_file():
            # 너무 긴 경로는 상대 경로로 잘라서 표기
            rel_path = item.relative_to(latest_dir)
            print(f"- {rel_path}")
else:
    print("No result directory to inspect.")